# Práctica — Clase 12: modelo del chatbot

Completa i due esercizi. Ognuno ha celle con `# TODO:` e una soluzione nascosta sotto.

In [ ]:
import tensorflow as tf
import numpy as np

pares = [
    ("hola", "inicio hola que tal"),
    ("como estas", "inicio bien gracias"),
    ("que es una red neuronal", "inicio un modelo inspirado en el cerebro"),
    ("adios", "inicio hasta pronto"),
]
preguntas = [p[0] for p in pares]
respuestas_objetivo = [p[1] + " fin" for p in pares]

LONGITUD_MAXIMA = 8
tokenizador = tf.keras.layers.TextVectorization(max_tokens=200, output_sequence_length=LONGITUD_MAXIMA)
tokenizador.adapt(preguntas + respuestas_objetivo)
vocabulario = tokenizador.get_vocabulary()
tamano_vocabulario = len(vocabulario)

entrada_encoder = tokenizador(preguntas).numpy()
secuencia_completa = tokenizador(respuestas_objetivo).numpy()
entrada_decoder = secuencia_completa[:, :-1]
objetivo_decoder = secuencia_completa[:, 1:]

## Esercizio 1 — Completa il codificatore-decodificatore

**Enunciato**: completa lo strato `LSTM` del codificatore affinché restituisca il suo stato (`return_state=True`), e passa quello stato come `initial_state` all'`LSTM` del decodificatore.

**Suggerimento**: una `LSTM` con `return_state=True` restituisce tre valori: l'output, lo stato nascosto e lo stato della cella.


In [ ]:
DIMENSION_EMBEDDING = 32
UNIDADES = 64

entrada_encoder_capa = tf.keras.Input(shape=(LONGITUD_MAXIMA,))
embedding_encoder = tf.keras.layers.Embedding(tamano_vocabulario, DIMENSION_EMBEDDING)(entrada_encoder_capa)
# TODO: crea una capa LSTM(UNIDADES, return_state=True) y aplícala a embedding_encoder
# guarda sus tres salidas como: _, estado_h, estado_c

entrada_decoder_capa = tf.keras.Input(shape=(LONGITUD_MAXIMA - 1,))
embedding_decoder = tf.keras.layers.Embedding(tamano_vocabulario, DIMENSION_EMBEDDING)(entrada_decoder_capa)
# TODO: crea una capa LSTM(UNIDADES, return_sequences=True, return_state=True)
# aplícala a embedding_decoder con initial_state=[estado_h, estado_c]

salida_final = tf.keras.layers.Dense(tamano_vocabulario, activation="softmax")(salida_decoder)
modelo = tf.keras.Model([entrada_encoder_capa, entrada_decoder_capa], salida_final)
modelo.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
modelo.fit([entrada_encoder, entrada_decoder], objetivo_decoder, epochs=100, verbose=0)
print("Entrenamiento terminado.")

<details>
<summary>💡 Vedi soluzione</summary>

```python
_, estado_h, estado_c = tf.keras.layers.LSTM(UNIDADES, return_state=True)(embedding_encoder)

salida_decoder, _, _ = tf.keras.layers.LSTM(UNIDADES, return_sequences=True, return_state=True)(
    embedding_decoder, initial_state=[estado_h, estado_c]
)
```
</details>

## Esercizio 2 — Completa la generazione avida

**Enunciato**: completa `generar_respuesta` affinché, a ogni passo, prenda il token con la probabilità più alta (`np.argmax`) e interrompa la generazione non appena quel token sia `"fin"`.

**Suggerimento**: `vocabulario[siguiente_token]` ti dà la parola corrispondente all'identificatore numerico.


In [ ]:
def generar_respuesta(pregunta):
    entrada = tokenizador([pregunta]).numpy()
    secuencia_generada = [vocabulario.index("inicio")]

    for _ in range(LONGITUD_MAXIMA - 1):
        entrada_actual = tf.keras.preprocessing.sequence.pad_sequences(
            [secuencia_generada], maxlen=LONGITUD_MAXIMA - 1, padding="post"
        )
        prediccion = modelo.predict([entrada, entrada_actual], verbose=0)
        # TODO: obtén siguiente_token con np.argmax sobre prediccion[0, len(secuencia_generada) - 1]
        # TODO: si vocabulario[siguiente_token] es "fin", detén el bucle (break)
        # TODO: si no, añade siguiente_token a secuencia_generada

    palabras = [vocabulario[t] for t in secuencia_generada[1:]]
    return " ".join(palabras)

print(generar_respuesta("hola"))

<details>
<summary>💡 Vedi soluzione</summary>

```python
prediccion = modelo.predict([entrada, entrada_actual], verbose=0)
siguiente_token = int(np.argmax(prediccion[0, len(secuencia_generada) - 1]))
if vocabulario[siguiente_token] == "fin":
    break
secuencia_generada.append(siguiente_token)
```
</details>